## Install Dependencies

In [1]:
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


## Load env variables

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-opus-4-6"


## Define Helper functions

In [37]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, prompt=None, model="claude-sonnet-4-5", temperature=0.5, stop_sequences=None):
    params = {
        "max_tokens": 1000,
        "messages": messages,
    }
    if prompt:
        params["system"] = prompt
    if temperature:
        params["temperature"] = temperature
    if model:
        params["model"] = model
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    response = client.messages.create(**params)
    if len(response.content) > 0:
        return response.content[0].text
    else:
        return ""

def request_user_input(messages):
    user_input = input("You: ")
    add_user_message(messages, user_input)
    return user_input

def chat_stream(messages, prompt=None, model="claude-sonnet-4-5", temperature=0.5, stop_sequences=None): 
    params = {
        "max_tokens": 1000,
        "messages": messages
    }
    if prompt:
        params["system"] = prompt
    if temperature:
        params["temperature"] = temperature
    if model:
        params["model"] = model
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    return client.messages.stream(**params)


## System Prompt

In [38]:
system_prompt = """
You are a AWS CLI expert.
You will be given a request from users and you will provide a bash command to solve the problem.
"""

## Evaluation Dataset

In [39]:
datasets = [
    {
        "input": "list all s3 buckets",
        "output": "aws s3 ls",
        "score": 0,
        "response": ""  
    },
    {
        "input": "find record on DynamoDB table my-table with id 123",
        "output": "aws dynamodb get-item --table-name my-table --key '{\"id\": {\"S\": \"123\"}}'",
        "score": 0,
        "response": ""
    },
    {
        "input": "check status on certificate for domain apiexample.com",
        "output": "aws acm list-certificates --domain apiexample.com",
        "score": 0,
        "response": ""
    }
]

## Test Prompt

In [41]:
for dataset in datasets:
    messages = []
    add_user_message(messages, dataset["input"])
    add_assistant_message(messages, "Here is your command in BASH\n```bash")
    with chat_stream(messages, prompt=system_prompt, temperature=0.0, stop_sequences=["```"]) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
        response = stream.get_final_message()
    if len(response.content) > 0:
        add_assistant_message(messages, response.content[0].text)
    dataset["response"] = response.content[0].text



aws s3 ls

aws dynamodb get-item \
    --table-name my-table \
    --key '{"id": {"N": "123"}}'

aws acm list-certificates --query "CertificateSummaryList[?DomainName=='apiexample.com']"


## Evaluator

In [42]:
import re

evaluator_prompt = """
You are a AWS CLI expert with the task of reviewing the response from the assistant and providing a score.
You will be given a response and the expected output.
Compare the response with the expected output and provide a concise score between 0 and 100, USING THE STRICT FORMAT OF ```Score: <score>```
Avoid any other text or comments.

Example:

Input:
```
    "expected_output": "aws s3 ls",
    "assistant_output": "aws s3 ls"
```

Output:
```
Score: 100
```
"""

def parse_score(text):
    match = re.search(r"Score:\s*(\d+)", text)
    return int(match.group(1)) if match else None

for dataset in datasets:
    messages = []
    dataset_result = f"""
    expected output: {dataset["output"]}
    assistant output: {dataset["response"]}
    """
    add_user_message(messages, dataset_result)
    response = chat(messages, prompt=evaluator_prompt, temperature=0.0)
    score = parse_score(response)
    dataset["score"] = score
    print(f"Input:    {dataset['input']}")
    print(f"Expected: {dataset['output']}")
    print(f"Got:      {dataset['response'].strip()}")
    print(f"Score:    {score}/100")
    print()


Input:    list all s3 buckets
Expected: aws s3 ls
Got:      aws s3 ls
Score:    100/100

Input:    find record on DynamoDB table my-table with id 123
Expected: aws dynamodb get-item --table-name my-table --key '{"id": {"S": "123"}}'
Got:      aws dynamodb get-item \
    --table-name my-table \
    --key '{"id": {"N": "123"}}'
Score:    30/100

Input:    check status on certificate for domain apiexample.com
Expected: aws acm list-certificates --domain apiexample.com
Got:      aws acm list-certificates --query "CertificateSummaryList[?DomainName=='apiexample.com']"
Score:    75/100



In [ ]:
## Results Summary
scores = [d["score"] for d in datasets if d["score"] is not None]
avg = sum(scores) / len(scores) if scores else 0
print(f"Evaluated {len(datasets)} prompts")
print(f"Average score: {avg:.1f}/100")
print(f"Scores: {scores}")